In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.impute import SimpleImputer

print("Libraries imported successfully!")

Libraries imported successfully!


In [22]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/titanic.csv")

print("First 5 rows of dataset:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nMissing values in each column:")
print(df.isnull().sum())

First 5 rows of dataset:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8

In [23]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/titanic.csv")

df_clean = df.copy()

df_clean = df_clean.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)

print("Columns after dropping irrelevant ones:")
print(df_clean.columns.tolist())

Columns after dropping irrelevant ones:
['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


In [24]:
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

print("Missing values after filling:")
print(df_clean.isnull().sum())

Missing values after filling:
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


In [25]:
le = LabelEncoder()

df_clean['Sex'] = le.fit_transform(df_clean['Sex'])

df_clean['Embarked'] = le.fit_transform(df_clean['Embarked'])

print("First 5 rows after converting text to numbers:")
print(df_clean.head())
print("\nValue mappings:")
print("Sex: male=1, female=0")
print("Embarked: C=0, Q=1, S=2")

First 5 rows after converting text to numbers:
   Survived  Pclass  Sex   Age  SibSp  Parch     Fare  Embarked
0         0       3    1  22.0      1      0   7.2500         2
1         1       1    0  38.0      1      0  71.2833         0
2         1       3    0  26.0      0      0   7.9250         2
3         1       1    0  35.0      1      0  53.1000         2
4         0       3    1  35.0      0      0   8.0500         2

Value mappings:
Sex: male=1, female=0
Embarked: C=0, Q=1, S=2


In [26]:
X = df_clean.drop('Survived', axis=1)

y = df_clean['Survived']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures:", X.columns.tolist())

Features shape: (891, 7)
Target shape: (891,)

Features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data size: {len(X_train)} rows")
print(f"Testing data size: {len(X_test)} rows")
print(f"\nTraining set - Survived: {sum(y_train)} Died: {len(y_train)-sum(y_train)}")
print(f"Testing set - Survived: {sum(y_test)} Died: {len(y_test)-sum(y_test)}")

Training data size: 712 rows
Testing data size: 179 rows

Training set - Survived: 268 Died: 444
Testing set - Survived: 74 Died: 105


In [28]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"Scaled training data shape: {X_train_scaled.shape}")
print("First row of scaled training data (first 5 values):")
print(X_train_scaled[0][:5])

Feature scaling completed!
Scaled training data shape: (712, 7)
First row of scaled training data (first 5 values):
[-1.61413602  0.7243102   1.25364106 -0.47072241 -0.47934164]


In [29]:
print("="*50)
print("LOGISTIC REGRESSION")
print("="*50)

lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)
print(f"Accuracy: {lr_accuracy:.3f} ({lr_accuracy*100:.1f}%)")

cm = confusion_matrix(y_test, lr_pred)
print("\nConfusion Matrix:")
print("          Predicted")
print("          Died  Survived")
print(f"Died      {cm[0,0]:5d}  {cm[0,1]:5d}")
print(f"Survived  {cm[1,0]:5d}  {cm[1,1]:5d}")

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', ascending=False)
print("\nFeature Importance (positive = increases survival chance):")
print(feature_importance.to_string(index=False))

LOGISTIC REGRESSION
Accuracy: 0.804 (80.4%)

Confusion Matrix:
          Predicted
          Died  Survived
Died         90     15
Survived     20     54

Feature Importance (positive = increases survival chance):
 Feature  Coefficient
    Fare     0.126216
   Parch    -0.098292
Embarked    -0.170860
   SibSp    -0.349582
     Age    -0.395574
  Pclass    -0.782061
     Sex    -1.278865


In [30]:
print("="*50)
print("SVM WITH DIFFERENT KERNELS")
print("="*50)

svm_results = {}

kernels = ['linear', 'rbf', 'poly']

for kernel in kernels:
    print(f"\n--- SVM with {kernel.upper()} kernel ---")

    svm_model = SVC(kernel=kernel, random_state=42)
    svm_model.fit(X_train_scaled, y_train)

    svm_pred = svm_model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, svm_pred)
    svm_results[kernel] = accuracy
    print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

SVM WITH DIFFERENT KERNELS

--- SVM with LINEAR kernel ---
Accuracy: 0.782 (78.2%)

--- SVM with RBF kernel ---
Accuracy: 0.816 (81.6%)

--- SVM with POLY kernel ---
Accuracy: 0.793 (79.3%)


In [31]:
print("="*50)
print("SVM WITH DIFFERENT C VALUES")
print("="*50)

C_values = [0.1, 1, 10, 100]

for C in C_values:
    print(f"\n--- SVM with C={C} (rbf kernel) ---")

    svm_model = SVC(kernel='rbf', C=C, random_state=42)
    svm_model.fit(X_train_scaled, y_train)

    svm_pred = svm_model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, svm_pred)
    print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

SVM WITH DIFFERENT C VALUES

--- SVM with C=0.1 (rbf kernel) ---
Accuracy: 0.810 (81.0%)

--- SVM with C=1 (rbf kernel) ---
Accuracy: 0.816 (81.6%)

--- SVM with C=10 (rbf kernel) ---
Accuracy: 0.799 (79.9%)

--- SVM with C=100 (rbf kernel) ---
Accuracy: 0.799 (79.9%)


In [32]:
print("="*50)
print("MODEL COMPARISON")
print("="*50)

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'SVM (linear)': SVC(kernel='linear', random_state=42),
    'SVM (rbf)': SVC(kernel='rbf', random_state=42),
    'SVM (poly)': SVC(kernel='poly', random_state=42),
    'SVM (rbf, C=10)': SVC(kernel='rbf', C=10, random_state=42)
}

results = {}

for name, model in models.items():

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy

    print(f"{name:25s}: {accuracy:.3f} ({accuracy*100:.1f}%)")

best_model_name = max(results, key=results.get)
best_accuracy = results[best_model_name]

print("\n" + "="*50)
print(f"BEST MODEL: {best_model_name}")
print(f"BEST ACCURACY: {best_accuracy:.3f} ({best_accuracy*100:.1f}%)")
print("="*50)

MODEL COMPARISON
Logistic Regression      : 0.804 (80.4%)
SVM (linear)             : 0.782 (78.2%)
SVM (rbf)                : 0.816 (81.6%)
SVM (poly)               : 0.793 (79.3%)
SVM (rbf, C=10)          : 0.799 (79.9%)

BEST MODEL: SVM (rbf)
BEST ACCURACY: 0.816 (81.6%)
